# 10 — Quantum Mutual Information

Lift mutual information to density matrices, and meet the first quantity of the course that reads correlation of every kind at once. You will compute $I(A{:}B) = S(A) + S(B) - S(AB)$ by hand from the eigenvalues of three $4\times4$ matrices. The correlated coins of `01/16` and the Bell pair of `01/17` come back side by side, as `01/19` announced, and part company: the coins share $1$ bit, exactly the classical ceiling of `02/03`, and the Bell pair shares $2$ bits. You will learn what that excess certifies (entanglement) and what it does not make of $I$ (an entanglement meter). Finally you will prove the property `02/03` promised: no channel applied to one side alone can increase $I(A{:}B)$.

**Prerequisites:** `02/03_mutual_information` (the three faces of $I$, the ceiling $I \le \min(H(X), H(Y))$, the data-processing inequality), `02/09_umegaki_relative_entropy` (the quantum relative entropy and its equality case), `02/02_kl_divergence` (§4(c): relative entropy never grows under processing), `01/13_pure_vs_mixed` (entropy from the spectrum), `01/16_tensor_and_partial_trace` (marginals by blocks; the correlated coins), `01/17_cnot_and_two_qubit_gates` (the Bell pair, CNOT), `01/18_entanglement` (pure whole and mixed parts; separable states; the partial-transpose test), `01/19_quantum_channels` (Kraus operators, the depolarizing channel, a channel as a unitary on system and environment).

**What you'll be able to do**
- Compute $S(A)$, $S(B)$ and $S(AB)$ from the eigenvalues of a two-qubit density matrix and its two marginals, and combine them into $I(A{:}B)$ and $S(A|B)$, by hand and with the library.
- Explain why the coins and the Bell pair, which have the same marginals and the same $Z$-statistics, share $1$ bit and $2$ bits.
- Say which step of the ceiling proof of `02/03` fails, state the bound that replaces it, and read the ceiling correctly.
- State and check the monotonicity of $I(A{:}B)$ under local channels, and recognise it as the quantum form of the data-processing inequality.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.infotheory.quantum import (
    quantum_conditional_entropy,
    quantum_mutual_information,
    quantum_relative_entropy,
)
from qot_course.quantum.composite import apply_channel, bell_state, depolarizing_channel, is_cptp, tensor
from qot_course.quantum.density import density_matrix, maximally_mixed, von_neumann_entropy
from qot_course.quantum.gates import CNOT, HADAMARD
from qot_course.quantum.states import KET_0, KET_1, KET_PLUS

rng = np.random.default_rng(10)   # random states of section 2 and random channels of section 3
viz.use_course_style()

## 1. Three entropies from three spectra

In `02/03` you wrote mutual information in three equivalent ways: as the relative entropy of a joint table to the product of its own marginals, as $H(X) + H(Y) - H(XY)$, and as $H(X) - H(X|Y)$. Each face has a quantum version, and the first two are the definition:

$$
I(A{:}B) \;=\; S(\rho_A) + S(\rho_B) - S(\rho_{AB}) \;=\; S(\rho_{AB}\,\|\,\rho_A\otimes\rho_B),
$$

where $\rho_A$ and $\rho_B$ are the partial traces of $\rho_{AB}$ (`01/16`) and $S(\rho\,\|\,\sigma)$ is the Umegaki relative entropy of `02/09`. The two forms agree for the same reason as classically. The logarithm of a tensor product splits, $\log(\rho_A\otimes\rho_B) = \log\rho_A\otimes I + I\otimes\log\rho_B$ (on the support, where the logarithms are defined), so $-\mathrm{tr}\,\rho_{AB}\log(\rho_A\otimes\rho_B) = S(\rho_A) + S(\rho_B)$, and the remaining term of the relative entropy is $-S(\rho_{AB})$. Like its classical parent, $I(A{:}B)$ is symmetric in $A$ and $B$, so it cannot say who drives whom (`02/04`). It is never negative, because it is a relative entropy, and it vanishes exactly when $\rho_{AB} = \rho_A\otimes\rho_B$ (the equality case of the relative entropy, `02/09`); these are the two properties `02/03` announced.

**The recipe by hand.** A von Neumann entropy is the Shannon entropy of the eigenvalues, $S(\rho) = -\sum_k \lambda_k\log_2\lambda_k$ (`01/13`). So you need three spectra: the spectrum of the $4\times4$ matrix $\rho_{AB}$ and those of its two $2\times2$ marginals. Three states, worked on paper:

- **A fair coin beside $|+\rangle$:** $\rho_{AB} = \tfrac{I}{2}\otimes|+\rangle\langle+|$. As a matrix it is $\tfrac14$ times two diagonal blocks $\begin{pmatrix}1&1\\1&1\end{pmatrix}$, and each block has eigenvalues $2$ and $0$. The spectrum is therefore $(\tfrac12, \tfrac12, 0, 0)$ and $S(AB) = 1$. The marginals are $\rho_A = I/2$, with $S(A) = 1$, and $\rho_B = |+\rangle\langle+|$, which is pure, with $S(B) = 0$. So $I = 1 + 0 - 1 = 0$.
- **The correlated coins:** $\rho_{AB} = \tfrac12|00\rangle\langle00| + \tfrac12|11\rangle\langle11| = \mathrm{diag}(\tfrac12, 0, 0, \tfrac12)$. It is diagonal, so its spectrum is $(\tfrac12, \tfrac12, 0, 0)$ again and $S(AB) = 1$. Both marginals are $I/2$, so $S(A) = S(B) = 1$ and $I = 1 + 1 - 1 = 1$ bit.
- **The Bell pair:** $\rho_{AB} = |\Phi^+\rangle\langle\Phi^+|$ has the coins' diagonal plus the two corners $\tfrac12$ at $|00\rangle\langle11|$ and $|11\rangle\langle00|$. On the span of $|00\rangle$ and $|11\rangle$ it is the block $\tfrac12\begin{pmatrix}1&1\\1&1\end{pmatrix}$, with eigenvalues $1$ and $0$. The spectrum is $(1, 0, 0, 0)$, the whole is pure, and $S(AB) = 0$. The marginals are $I/2$, as `01/18` computed by blocks. So $I = 1 + 1 - 0 = 2$ bits.

The first two states have the same global spectrum; their mutual information differs only because their parts differ. Correlation lives in how the whole compares with its parts, never in the whole alone. The cell below repeats these computations with `numpy`, then calls the library, evaluates the relative-entropy face, and prints the conditional entropy $S(A|B) = S(AB) - S(B)$ beside each line.

In [ ]:
def tidy(v):
    # round away floating-point dust such as -1e-16, so that a zero prints as 0.000 and not -0.000
    return float(np.round(v, 12)) + 0.0

def entropy_from_eigenvalues(eigs):
    # S = -sum_k lambda_k log2 lambda_k, with 0 log 0 = 0: a zero eigenvalue carries no uncertainty
    eigs = np.clip(np.real(eigs), 0.0, None)
    nonzero = eigs[eigs > 1e-12]
    return tidy(-np.sum(nonzero * np.log2(nonzero)))

def marginals_by_blocks(rho):
    # 01/16: entry (a, a') of rho_A is the trace of block (a, a'); rho_B is the sum of the diagonal blocks
    blocks = rho.reshape(2, 2, 2, 2)                    # indices (a, b, a', b')
    return np.einsum("abcb->ac", blocks), np.einsum("abad->bd", blocks)

states = {
    "coin beside |+>": tensor(maximally_mixed(2), density_matrix(KET_PLUS)),
    "correlated coins": 0.5 * density_matrix(tensor(KET_0, KET_0)) + 0.5 * density_matrix(tensor(KET_1, KET_1)),
    "Bell pair": density_matrix(bell_state()),
}

results = {}
for name, rho in states.items():
    rho_A, rho_B = marginals_by_blocks(rho)
    eig_AB, eig_A, eig_B = np.linalg.eigvalsh(rho), np.linalg.eigvalsh(rho_A), np.linalg.eigvalsh(rho_B)
    S_A, S_B, S_AB = (entropy_from_eigenvalues(e) for e in (eig_A, eig_B, eig_AB))
    I_hand = S_A + S_B - S_AB
    results[name] = dict(S_A=S_A, S_B=S_B, S_AB=S_AB, I=I_hand, cond=S_AB - S_B)
    print(f"{name}")
    print(f"   eigenvalues of rho_AB = {np.round(eig_AB[::-1], 3) + 0.0}   of rho_A = {np.round(eig_A[::-1], 3) + 0.0}   of rho_B = {np.round(eig_B[::-1], 3) + 0.0}")
    print(f"   by hand:  S(A) = {S_A:.3f}   S(B) = {S_B:.3f}   S(AB) = {S_AB:.3f}   ->  I(A:B) = {I_hand:.3f} bit   S(A|B) = {S_AB - S_B:+.3f} bit")
    print(f"   library:  I(A:B) = {tidy(quantum_mutual_information(rho, dims=[2, 2])):.3f}   S(A|B) = {tidy(quantum_conditional_entropy(rho, dims=[2, 2])):+.3f}"
          f"   relative-entropy face S(rho || rho_A x rho_B) = {tidy(quantum_relative_entropy(rho, np.kron(rho_A, rho_B))):.3f}")

names = list(results)
x = np.arange(len(names))
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4), gridspec_kw={"wspace": 0.28})
ax = axes[0]
for k, (key, label, colour) in enumerate([("S_A", "S(A)", viz.SOURCE_COLOR), ("S_B", "S(B)", viz.TARGET_COLOR), ("S_AB", "S(AB)", COLORS["muted"])]):
    values = [results[n][key] for n in names]
    ax.bar(x + (k - 1) * 0.26, values, width=0.26, color=colour, label=label)
    for xi, v in zip(x + (k - 1) * 0.26, values):   # value labels, so that a zero entropy is visible
        ax.text(xi, v + 0.03, f"{v:.0f}", ha="center", fontsize=10, color=COLORS["text"])
ax.set_xticks(x, names); ax.set_ylabel("entropy (bits)"); ax.set_ylim(0, 1.35)
ax.set_title("The three entropies, from the eigenvalues", pad=10); ax.legend(loc="upper left", ncol=3)
ax = axes[1]
ax.bar(x, [results[n]["I"] for n in names], width=0.5, color=COLORS["quantum"], label="I(A:B) = S(A) + S(B) - S(AB)")
for k, n in enumerate(names):
    ceiling = min(results[n]["S_A"], results[n]["S_B"])
    ax.hlines(ceiling, k - 0.34, k + 0.34, color=COLORS["highlight"], lw=3, label="classical ceiling min(S(A), S(B))" if k == 0 else None)
    cond = results[n]["cond"]
    cond_text = "0" if abs(cond) < 1e-9 else f"{cond:+.0f}"
    ax.text(k, max(results[n]["I"], ceiling) + 0.08, f"I = {results[n]['I']:.0f},  S(A|B) = {cond_text}", ha="center", fontsize=10, color=COLORS["text"])
ax.set_xticks(x, names); ax.set_ylabel("I(A:B)  (bits)"); ax.set_ylim(0, 2.5)
ax.set_title("Mutual information against each state's ceiling", pad=10); ax.legend(loc="upper left")
plt.show()

**Read the output and the figure.** Every spectrum printed is the one found on paper. The hand computation, the library and the relative-entropy face agree on every line: $I = 0$, $1$ and $2$ bits.

On the left, the coins and the Bell pair have identical bars for $S(A)$ and $S(B)$, one bit each, because their marginals are both $I/2$. They differ in the third bar. The coins' whole is as uncertain as each part ($S(AB) = 1$), while the Bell pair's whole is certain ($S(AB) = 0$): "pure whole, mixed parts", the row of `01/18` that no classical table has.

On the right, the rose dash on each bar is the classical ceiling $\min(S(A), S(B))$ of `02/03`, computed from that state's own marginals. The coin beside $|+\rangle$ shares nothing. The coins touch their ceiling exactly, with $S(A|B) = 0$: once you know $B$, nothing about $A$ is left uncertain, and that is the equality case of `02/03`. The Bell pair stands at twice its ceiling, with $S(A|B) = -1$. That negative number is where the classical proof breaks, as section 2 shows. Welcome it: this is the first place in the course where the quantum lift does something no classical table can.

## 2. The classical ceiling, and the step that breaks

This is the densest section, and the place where your classical intuition grows: each step reuses a proof you already own. Return to the proof in `02/03` §2. Classically, $I(X;Y) = H(X) - H(X|Y)$, and $H(X|Y) \ge 0$ because it is an average of Shannon entropies. Hence $I \le H(X)$, by symmetry $I \le H(Y)$, and so $I \le \min(H(X), H(Y))$. The first step survives the lift to density matrices word for word. Rearranging the definition gives the identity

$$
I(A{:}B) = S(A) - S(A|B), \qquad S(A|B) = S(AB) - S(B),
$$

for every state. The second step is the one that fails. The quantum conditional entropy is not an average of entropies of anything, and it can be negative: the Bell pair has $S(A|B) = -1$. With $S(A|B) < 0$ the identity puts $I$ above $S(A)$. The meaning of a negative conditional entropy is the subject of `02/11_negative_conditional_entropy`; here you need only its sign.

**How far above?** The Araki–Lieb inequality $S(AB) \ge |S(A) - S(B)|$ (Araki & Lieb 1970), which you will prove yourself in exercise 4(b), caps the damage, since it gives $S(A|B) \ge -S(A)$ and $S(B|A) \ge -S(B)$. Substituting into the identity,

$$
I(A{:}B) \;\le\; 2\min\big(S(A), S(B)\big) \;\le\; 2\log_2 d,
$$

where $d = \min(\dim A, \dim B)$; this is twice the classical ceiling. For two qubits it is $2$ bits, the Bell pair's value. Pure states sit exactly on this bound: a pure whole has $S(AB) = 0$ and two parts with the same spectrum (the Schmidt decomposition, Nielsen & Chuang §2.5), so $I = 2S(A)$. That is twice the entanglement entropy of `01/18`. It exceeds the classical ceiling exactly when the state is entangled, since $2S(A) > S(A)$ whenever $S(A) > 0$.

**What the excess certifies for mixed states.** A separable state, a mixture of products (`01/18`), obeys a theorem we quote, as `02/09` quoted Lindblad's: for a separable state the whole is at least as uncertain as each part, $S(AB) \ge S(A)$ and $S(AB) \ge S(B)$. (For the curious: Nielsen & Kempe (2001) prove it by showing that the global spectrum is majorized by each local one and that entropy is Schur-concave; nothing below relies on that route.) Both conditional entropies of a separable state are therefore non-negative, the classical proof goes through, and

$$
\rho_{AB}\ \text{separable} \;\Longrightarrow\; I(A{:}B) \le \min\big(S(A), S(B)\big).
$$

Read it in the contrapositive: **a value above the classical ceiling proves that the state is entangled.** The converse does not hold, and that matters as much. A value at or below the ceiling proves nothing. The correlated coins sit exactly on the ceiling and are separable by construction. Mix the Bell pair half-and-half with the maximally mixed state $I/4$ and you get a state that is still entangled, as its partial transpose shows, yet whose mutual information falls below the ceiling. Its spectrum is $(0.625, 0.125, 0.125, 0.125)$, so $S(AB) = 1.549$, its marginals are still $I/2$, and $I = 1 + 1 - 1.549 = 0.451 < 1$ bit; the cell prints each of these numbers.

The cell below draws random two-qubit states (pure, mixed of rank $2$ to $4$, and mixtures of products) and plots $I(A{:}B)$ against each state's own ceiling $\min(S(A), S(B))$. It colours each mixed state by the partial-transpose test of `01/18`, exact for two qubits (Peres 1996; Horodecki, Horodecki & Horodecki 1996), written inline as in `01/18` because the library has none.

In [ ]:
def partial_transpose_B(rho):
    # 01/18: transpose each 2x2 block in place, (rho^{T_B})_{ab,a'b'} = rho_{ab',a'b}
    return rho.reshape(2, 2, 2, 2).transpose(0, 3, 2, 1).reshape(4, 4)

def random_qubit():
    v = rng.normal(size=2) + 1j * rng.normal(size=2)
    return v / np.linalg.norm(v)

def random_state(rank):
    # rho = G G^dagger / tr, with G a 4 x rank matrix of complex Gaussians: rank 1 is a random pure state
    G = rng.normal(size=(4, rank)) + 1j * rng.normal(size=(4, rank))
    rho = G @ G.conj().T
    return rho / np.trace(rho).real

def random_separable(n_terms):
    # a mixture of products: separable by construction (01/18)
    weights = rng.dirichlet(np.ones(n_terms))
    return sum(w * density_matrix(np.kron(random_qubit(), random_qubit())) for w in weights)

def ceiling_and_qmi(rho):
    rho_A, rho_B = marginals_by_blocks(rho)
    return min(von_neumann_entropy(rho_A), von_neumann_entropy(rho_B)), quantum_mutual_information(rho, dims=[2, 2])

def is_entangled(rho):
    # two qubits: a negative eigenvalue of the partial transpose <=> entangled (Peres; Horodecki, 01/18)
    return np.linalg.eigvalsh(partial_transpose_B(rho)).min() < -1e-10

pure = np.array([ceiling_and_qmi(random_state(1)) for _ in range(300)])
mixed_states = [random_state(r) for r in (2, 3, 4) for _ in range(300)] + [random_separable(int(rng.integers(1, 5))) for _ in range(300)]
mixed = np.array([ceiling_and_qmi(r) for r in mixed_states])
entangled = np.array([is_entangled(r) for r in mixed_states])
above = mixed[:, 1] > mixed[:, 0] + 1e-9

print(f"pure states:     {len(pure)}   largest |I - 2 S(A)| = {np.abs(pure[:, 1] - 2 * pure[:, 0]).max():.1e}")
print(f"mixed states:    {len(mixed)}   separable {np.sum(~entangled)} (of which 300 drawn as mixtures of products), entangled {np.sum(entangled)}")
print(f"   separable states above the classical ceiling: {np.sum(above & ~entangled)}")
print(f"   entangled states above the ceiling: {np.sum(above & entangled)}   below it: {np.sum(~above & entangled)}")
all_points = np.vstack([pure, mixed])
print(f"largest I - 2 min(S(A), S(B)) over all {len(all_points)} states = {np.max(all_points[:, 1] - 2 * all_points[:, 0]):.1e}   (Araki-Lieb: never positive)")
werner_half = 0.5 * states["Bell pair"] + 0.5 * np.eye(4) / 4
eig_half = np.linalg.eigvalsh(werner_half)[::-1]
print(f"Bell pair mixed half-and-half with I/4:  spectrum = {np.round(eig_half, 3)},  S(AB) = {entropy_from_eigenvalues(eig_half):.3f},  by hand I = 1 + 1 - S(AB) = {2 - entropy_from_eigenvalues(eig_half):.3f}")
print(f"Bell pair mixed half-and-half with I/4:  I = {quantum_mutual_information(werner_half, dims=[2, 2]):.3f} bit,  ceiling = {ceiling_and_qmi(werner_half)[0]:.3f},"
      f"  smallest eigenvalue of the partial transpose = {np.linalg.eigvalsh(partial_transpose_B(werner_half)).min():+.3f}")

fig, ax = plt.subplots(figsize=(8.5, 6.6))
xs = np.linspace(0, 1, 2)
ax.fill_between(xs, xs, 2 * xs, color=COLORS["highlight"], alpha=0.12, label="above the ceiling: entangled states only")
ax.plot(xs, xs, color=COLORS["highlight"], lw=2, ls="--", label="classical ceiling  I = min(S(A), S(B))")
ax.plot(xs, 2 * xs, color=COLORS["muted"], lw=1.5, ls=":", label="Araki-Lieb bound  I = 2 min(S(A), S(B))")
ax.scatter(*mixed[~entangled].T, s=9, color=viz.FLOW_COLOR, alpha=0.7, label="separable states")
ax.scatter(*mixed[entangled].T, s=9, color=viz.SOURCE_COLOR, alpha=0.8, label="entangled mixed states")
ax.scatter(*pure.T, s=9, color=COLORS["quantum"], alpha=0.9, label="pure states")
for name, text_at in zip(names, [(0.10, 0.62), (1.03, 0.93), (1.03, 2.03)]):
    c, i = min(results[name]["S_A"], results[name]["S_B"]), results[name]["I"]
    ax.scatter([c], [i], s=70, color=COLORS["text"], zorder=5)
    ax.annotate(name, (c, i), text_at, fontsize=10.5, color=COLORS["text"],
                arrowprops=dict(arrowstyle="->", color=COLORS["text"], lw=1))
ax.set_xlabel("classical ceiling  min(S(A), S(B))  (bits)"); ax.set_ylabel("I(A:B)  (bits)")
ax.set_xlim(-0.03, 1.3); ax.set_ylim(-0.05, 2.15)
ax.set_title("Who can sit above the classical ceiling?", pad=12); ax.legend(loc="upper left", fontsize=9.5)
plt.show()

**Read the output and the figure.** The pure states (blue) lie on the dotted line $I = 2\min(S(A), S(B))$: over $300$ random pure states, $I$ and $2S(A)$ differ by at most $2.4\times10^{-15}$. No state of the $1500$ drawn rises above that line, which is the Araki–Lieb bound at work. In this run (seed $10$, cells run in order), $389$ of the $1200$ mixed states are separable and $811$ entangled. These counts describe the sampler, not nature: $300$ of the separable ones are mixtures of products by construction. The output counts $0$ separable states in the band, and the separable states (green) all stay on or below the dashed ceiling. That is a consistency check of the quoted theorem, not evidence for it. Your classical ceiling has not been thrown away; it has been promoted to an entanglement test.

The entangled mixed states (violet) are on both sides, in this run $225$ above the ceiling and $586$ below it. The Bell pair mixed half-and-half with $I/4$ has $I = 0.451$ bit, below its ceiling of $1$ bit, while the smallest eigenvalue of its partial transpose is $-0.125$, so it is entangled. The labelled points are the states of section 1.

**Total correlation, not an entanglement meter.** $I(A{:}B)$ counts every correlation between $A$ and $B$, classical and quantum together; Groisman, Popescu & Winter (2005) showed it is the amount of noise needed to erase all of them. The coins have $1$ bit of it and no entanglement, a noisy Bell pair has less with entanglement intact, and this total correlation is what Module 04 reads as coupling.

## 3. Local processing never increases it

`02/03` §3 proved the data-processing inequality: if $Z$ is made from $Y$ alone, $I(X;Z) \le I(X;Y)$. It also announced that the quantum mutual information obeys the same rule. Here it is.

**Monotonicity under local channels.** Let $\Phi$ be any channel (a completely positive, trace-preserving map, `01/19`) acting on $B$ alone, and write $\rho_{AB'} = (\mathrm{id}\otimes\Phi)(\rho_{AB})$. Then

$$
I(A{:}B') \;\le\; I(A{:}B),
$$

and the same holds for a channel on $A$. The proof takes two lines once you use the relative-entropy face of section 1. First, the Umegaki relative entropy never increases under a channel, $S(\mathcal E(\rho)\,\|\,\mathcal E(\sigma)) \le S(\rho\,\|\,\sigma)$ (Lindblad 1975; Uhlmann 1977); this is the quantum version of `02/02` §4(c), which `02/02` announced for the quantum lift. Apply it with $\mathcal E = \mathrm{id}\otimes\Phi$, which is itself a channel because $\Phi$ is *completely* positive (`01/19`), to $\rho = \rho_{AB}$ and $\sigma = \rho_A\otimes\rho_B$. Second, $\mathcal E$ sends the product of the marginals to $\rho_A\otimes\Phi(\rho_B)$, which is exactly the product of the marginals of the output. So

$$
I(A{:}B') = S\big(\rho_{AB'}\,\|\,\rho_A\otimes\Phi(\rho_B)\big) = S\big(\mathcal E(\rho_{AB})\,\|\,\mathcal E(\rho_A\otimes\rho_B)\big) \le S(\rho_{AB}\,\|\,\rho_A\otimes\rho_B) = I(A{:}B).
$$

A local unitary $U$ keeps $I$ unchanged, because $U^\dagger$ undoes it and the inequality must then hold in both directions. The word *local* carries the whole statement. A gate acting on both qubits at once is not covered, and CNOT turns the product $|+\rangle|0\rangle$, with $I = 0$, into the Bell pair, with $I = 2$ bits (`01/17`).

The cell below checks the statement four ways. First, it applies the library's depolarizing channel (`01/19`) to $B$ alone, with strength $p$ from $0$ to $1$, on the three states of section 1 and the tilted pure state $\cos\tfrac\pi8|00\rangle + \sin\tfrac\pi8|11\rangle$. On the coins it flips a classical bit with probability $p/2$, the binary symmetric channel of `02/03`, so $I$ should follow $1 - h(p/2)$. Second, it builds $2000$ random channels as `01/19` derived them (a random unitary on $B$ and a fresh environment qubit, then a trace over the environment) and applies each to $B$ of a random state. Third, it applies the full dephasing of `01/19` to $B$ of the Bell pair. Fourth, it applies a local unitary and a global gate.

In [ ]:
def on_B(kraus):
    # a channel acting on B alone: each Kraus operator becomes I (x) K
    return [np.kron(np.eye(2), K) for K in kraus]

def h2(p):
    p = np.clip(p, 1e-15, 1 - 1e-15)
    return -p * np.log2(p) - (1 - p) * np.log2(1 - p)

theta = np.pi / 8
tilted = density_matrix(np.cos(theta) * np.kron(KET_0, KET_0) + np.sin(theta) * np.kron(KET_1, KET_1))
family = {"Bell pair": states["Bell pair"], "correlated coins": states["correlated coins"],
          "tilted pure state, theta = pi/8": tilted, "coin beside |+>": states["coin beside |+>"]}
ps = np.linspace(0.0, 1.0, 41)
curves = {name: np.array([quantum_mutual_information(apply_channel(rho, on_B(depolarizing_channel(p))), dims=[2, 2]) for p in ps])
          for name, rho in family.items()}
print(f"I(A:B) after depolarizing B with p = 0, 0.5, 1:")
for name, c in curves.items():
    print(f"   {name:32s} {tidy(c[0]):.3f} -> {tidy(c[20]):.3f} -> {tidy(c[-1]):.3f}    largest step up along the sweep = {np.max(np.diff(c)):.1e}")
print(f"coins: largest |I - (1 - h(p/2))| along the sweep = {np.max(np.abs(curves['correlated coins'] - (1 - h2(ps / 2)))):.1e}   (02/03's flip channel with flip probability p/2)")

def random_channel_by_dilation():
    # 01/19: a unitary on B and a fresh environment qubit, then trace the environment out
    G = rng.normal(size=(4, 4)) + 1j * rng.normal(size=(4, 4))
    Q, R = np.linalg.qr(G)
    U = (Q * (np.diag(R) / np.abs(np.diag(R)))).reshape(2, 2, 2, 2)   # indices (b, e, b', e')
    return [U[:, e, :, 0] for e in range(2)]                           # K_e = <e|_E U |0>_E

changes = []
for _ in range(2000):
    rho = random_state(int(rng.integers(1, 5)))
    kraus = random_channel_by_dilation()
    assert is_cptp(kraus)
    changes.append(quantum_mutual_information(apply_channel(rho, on_B(kraus)), dims=[2, 2]) - quantum_mutual_information(rho, dims=[2, 2]))
changes = np.array(changes)
print(f"2000 random states, each through a random channel on B: largest change of I(A:B) = {changes.max():.1e}   (never positive)")
full_dephasing = [np.diag([1.0, 0.0]).astype(complex), np.diag([0.0, 1.0]).astype(complex)]   # 01/19's dephasing with lambda = 1
bell_dephased = apply_channel(states["Bell pair"], on_B(full_dephasing))
print(f"full dephasing of B (01/19) on the Bell pair: I goes from 2.000 to {tidy(quantum_mutual_information(bell_dephased, dims=[2, 2])):.3f}"
      f"   output equals the correlated coins: {np.allclose(bell_dephased, states['correlated coins'])}")
bell_rotated = apply_channel(states["Bell pair"], on_B([HADAMARD]))
print(f"a local unitary (Hadamard on B) on the Bell pair: I = {tidy(quantum_mutual_information(bell_rotated, dims=[2, 2])):.3f}   (reversible, so nothing lost)")
product = density_matrix(np.kron(KET_PLUS, KET_0))
print(f"a GLOBAL gate (CNOT) on |+>|0>: I goes from {tidy(quantum_mutual_information(product, dims=[2, 2])):.3f} to {tidy(quantum_mutual_information(CNOT @ product @ CNOT.conj().T, dims=[2, 2])):.3f}   (not local: no promise)")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4), gridspec_kw={"wspace": 0.28})
ax = axes[0]
for (name, c), colour in zip(curves.items(), [COLORS["quantum"], viz.FLOW_COLOR, viz.SOURCE_COLOR, COLORS["muted"]]):
    ax.plot(ps, c, color=colour, lw=2.4, label=name)
ax.plot(ps, 1 - h2(ps / 2), color=COLORS["text"], lw=1, ls=":", label="1 - h(p/2): 02/03's flip channel")
ax.set_xlabel("depolarizing strength p, applied to B only"); ax.set_ylabel("I(A:B)  (bits)")
ax.set_title("Noise on one side only ever lowers I(A:B)", pad=10); ax.legend(loc="upper right", fontsize=9.5)
ax = axes[1]
ax.hist(changes, bins=40, color=viz.SOURCE_COLOR, alpha=0.85)
ax.axvline(0.0, color=COLORS["highlight"], lw=2, ls="--", label="no change")
ax.set_xlabel("I(A:B) after  -  I(A:B) before  (bits)"); ax.set_ylabel("number of trials")
ax.set_title("2000 random states through random local channels", pad=10); ax.legend(loc="upper left")
plt.show()

**Read the output and the figure.** On the left, every curve falls as the noise on $B$ grows, and none of them ever turns up: along each sweep the largest step is negative, or zero to rounding for the coin beside $|+\rangle$, which has nothing to lose. The Bell pair goes from $2$ bits to $0.451$ at $p = 0.5$ and to $0$ at $p = 1$. The output at $p = 0.5$ is the half-and-half mixture of section 2. The coins go from $1$ bit to $0.189$ and then $0$. Their curve lies on the dotted $1 - h(p/2)$ to within $5.1\times10^{-14}$: on a diagonal state, the quantum statement is the classical data-processing inequality of `02/03`. The tilted pure state starts at $1.202$ bits, which is $2S(A)$ as section 2 predicts for a pure state.

The full dephasing of $B$ turns the Bell pair into the correlated coins of `01/19`, and the output prints $I$ going from $2$ bits to $1.000$: one local channel takes away one of the two bits and leaves the bit that the coins share.

On the right, in this run every one of the $2000$ changes of $I$ is negative; the largest is $-3.6\times10^{-3}$ bit. That no draw reached zero is a reading of this plot; the theorem itself only guarantees $\le 0$. Equality comes with reversible maps: a Hadamard on $B$ leaves the Bell pair at $2.000$ bits. CNOT, a gate acting on both qubits, raises $I$ from $0$ to $2$ bits, and the inequality says nothing about it. Correlation can be created only by interaction between the parts, or by a source they share; each party acting alone can keep it or lose it, never add to it.

For Module 04 this is the guarantee that matters. A channel applied to one half of an embedded state can hide coupling, but it cannot make the two halves look more coupled than they were.

## Your turn

1. **The other Bell states.** Build $|\Psi^+\rangle = (I\otimes X)|\Phi^+\rangle = \tfrac{1}{\sqrt2}(|01\rangle + |10\rangle)$. Write its $4\times4$ density matrix, find its spectrum and its two marginals by hand, and compute $I(A{:}B)$ with `entropy_from_eigenvalues` and with `quantum_mutual_information`. Then explain, using section 3 alone, why every state of the form $(U\otimes V)|\Phi^+\rangle$ with $U$, $V$ unitary must share exactly $2$ bits.
2. **Diagonal states are classical tables.** Build the diagonal density matrix of the $90\%$ table of `02/03`, $\mathrm{diag}(0.45, 0.05, 0.05, 0.45)$, and the anti-correlated coins $\mathrm{diag}(0, \tfrac12, \tfrac12, 0)$. Compute $I(A{:}B)$ for both with `quantum_mutual_information`, and compare with `mutual_information` from `qot_course.infotheory.classical` applied to the $2\times2$ tables. Explain in two sentences why a diagonal $\rho_{AB}$ always gives the classical value.
3. **Which ceiling?** `02/11` sweeps the family $\rho(p) = (1-p)\,|\Phi^+\rangle\langle\Phi^+| + p\,I/4$. (a) Show by hand that $\rho_A = \rho_B = I/2$ for every $p$, so $S(A) = 1$ bit along the whole family, and use the identity $I = S(A) - S(A|B)$ of section 2 to explain why $I$ falls through $1$ bit at the same $p$ at which $S(A|B)$ climbs through $0$. (b) Mix the tilted pure state of section 3 in the same way, and find numerically where $I$ falls through $1$ bit and where it falls through its own ceiling $\min(S(A), S(B))$. Which crossing coincides with $S(A|B) = 0$, and what was special about the family in (a)? Say in one sentence why $I$ must be compared with the state's own ceiling, never with a fixed number of bits.
4. **Harder: the nearest product, and the bound of section 2.** (a) For any product state $\sigma_A\otimes\sigma_B$, prove by hand that $S(\rho_{AB}\,\|\,\sigma_A\otimes\sigma_B) = I(A{:}B) + S(\rho_A\,\|\,\sigma_A) + S(\rho_B\,\|\,\sigma_B)$. (Split the logarithm as in section 1 and take partial traces.) Conclude that $I(A{:}B)$ is the relative-entropy distance from $\rho_{AB}$ to the *nearest* product state, attained at $\rho_A\otimes\rho_B$: the "distance to the product family" that `02/05` announced. Check the identity on a random state and twenty random products. (b) Prove the Araki–Lieb inequality $S(AB) \ge S(A) - S(B)$. *Hint:* write $\rho_{AB} = \sum_k \lambda_k |k\rangle\langle k|$, add a system $C$ with orthonormal states $|k\rangle_C$, and check that $|\psi\rangle = \sum_k \sqrt{\lambda_k}\,|k\rangle_{AB}|k\rangle_C$ is pure and that tracing out $C$ returns $\rho_{AB}$. By section 2 the two sides of a pure state have equal entropies, so $S(A) = S(BC)$ and $S(C) = S(AB)$. Finish with subadditivity $S(BC) \le S(B) + S(C)$, which is $I(B{:}C) \ge 0$ from section 1. Suggested extension: repeat the sweep of section 3 with the amplitude-damping channel of `01/19` on $A$.

## What you built

- You can compute the quantum mutual information by hand from the spectra of $\rho_{AB}$, $\rho_A$ and $\rho_B$: $0$ for a coin beside $|+\rangle$, $1$ bit for the correlated coins, $2$ bits for the Bell pair, in agreement with the library and with the face $S(\rho_{AB}\,\|\,\rho_A\otimes\rho_B)$.
- You can explain why the coins and the Bell pair, with the same marginals, share $1$ and $2$ bits: the difference is the whole, mixed for the coins and pure for the Bell pair.
- You can name the step of `02/03`'s proof that fails, $H(X|Y) \ge 0$, and state the bound $I \le 2\min(S(A), S(B))$ that replaces it.
- You can read the classical ceiling: above it means entangled, at or below it means nothing.
- You can say what $I(A{:}B)$ is, total correlation with classical and quantum parts together, and what it is not, an entanglement measure for mixed states.
- You can prove that no channel on one side increases $I(A{:}B)$, from the monotonicity of the Umegaki relative entropy, and you checked it on the depolarizing channel and on $2000$ random channels. On a diagonal state it reduces to the data-processing inequality of `02/03`.

The quantum lift of mutual information is now yours, with its surprise and its limits in plain view.

## What's next

The step that broke the ceiling deserves a notebook of its own. In `02/11_negative_conditional_entropy` you will meet the quantity $S(A|B)$ that went negative for the Bell pair, learn what a negative conditional entropy means, and sweep the noisy Bell family to watch it change sign.

## References

- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, 10th anniversary ed., Cambridge University Press (2010) — §2.5 (Schmidt decomposition: the two sides of a pure state share one spectrum), ch. 11 (quantum mutual information and conditional entropy, subadditivity). doi:10.1017/CBO9780511976667
- M. M. Wilde, *Quantum Information Theory*, 2nd ed., Cambridge University Press (2017) — ch. 11 (quantum mutual information and its properties). doi:10.1017/9781316809976
- H. Araki & E. H. Lieb, *Entropy inequalities*, Commun. Math. Phys. **18**, 160–170 (1970) — the inequality $S(AB) \ge |S(A) - S(B)|$ behind $I \le 2\min(S(A), S(B))$. doi:10.1007/BF01646092
- M. A. Nielsen & J. Kempe, *Separable states are more disordered globally than locally*, Phys. Rev. Lett. **86**, 5184–5187 (2001) — for separable states $S(AB) \ge S(A)$ and $S(AB) \ge S(B)$, hence the classical ceiling. doi:10.1103/PhysRevLett.86.5184
- B. Groisman, S. Popescu & A. Winter, *Quantum, classical, and total amount of correlations in a quantum state*, Phys. Rev. A **72**, 032317 (2005) — the quantum mutual information as the total amount of correlation. doi:10.1103/PhysRevA.72.032317
- G. Lindblad, *Completely positive maps and entropy inequalities*, Commun. Math. Phys. **40**, 147–151 (1975) — monotonicity of the relative entropy under channels. doi:10.1007/BF01609396
- A. Uhlmann, *Relative entropy and the Wigner–Yanase–Dyson–Lieb concavity in an interpolation theory*, Commun. Math. Phys. **54**, 21–32 (1977) — monotonicity in full generality. doi:10.1007/BF01609834
- A. Peres, *Separability criterion for density matrices*, Phys. Rev. Lett. **77**, 1413–1415 (1996) — the partial-transpose test. doi:10.1103/PhysRevLett.77.1413
- M. Horodecki, P. Horodecki & R. Horodecki, *Separability of mixed states: necessary and sufficient conditions*, Phys. Lett. A **223**, 1–8 (1996) — the test is exact for two qubits. doi:10.1016/S0375-9601(96)00706-2

**Previous:** `notebooks/02_InformationAndGeometry/09_umegaki_relative_entropy.ipynb` · **Next:** `notebooks/02_InformationAndGeometry/11_negative_conditional_entropy.ipynb`